# Open-Vocabulary Vision -- CLIP

Train an image encoder and a text encoder together so that matching (image, caption) pairs land at the same point in a shared space. That is the whole trick.

## Problem Definition

Traditional classifiers are closed-vocabulary: a 1000-class ImageNet model can only predict 1000 labels. Every new category requires labelled data and a retrained head.


Classify into any set of categories at inference, described purely in natural language. You give it a new class by writing a sentence.

## Basic Concept


### Two towers

```mermaid
flowchart LR
    IMG["Image"] --> IENC["Image encoder<br/>(ViT-L/14)"] --> IEMB["Image embedding<br/>(1024,)"]
    TXT["Caption"] --> TENC["Text encoder<br/>(transformer)"] --> TEMB["Text embedding<br/>(1024,)"]
    IEMB --> SIM["Cosine similarity"]
    TEMB --> SIM

    style IENC fill:#dbeafe,stroke:#2563eb
    style TENC fill:#fef3c7,stroke:#d97706
    style SIM fill:#dcfce7,stroke:#16a34a
```

Both encoders end with a linear projection to the **same embedding dimension** (512 for CLIP-B/32, 1024 for CLIP-L/14). **L2-normalise and compute cosine similarity.**

### The objective

Given a batch of N (image, caption) pairs, build an NxN similarity matrix. Train both encoders so the diagonal (matching pairs) has high similarity and off-diagonals (non-matching) have low similarity.

```
sim_matrix = image_embeddings @ text_embeddings.T / tau

loss_i2t = cross_entropy(sim_matrix,       targets=arange(N))
loss_t2i = cross_entropy(sim_matrix.T,     targets=arange(N))
loss = (loss_i2t + loss_t2i) / 2
```

Symmetric because both image-to-text and text-to-image retrieval should work. `tau` (temperature) is typically learned as a scalar parameter, initialised to 0.07.

### SigLIP: a better loss

SigLIP (Zhai et al., 2023) replaced the softmax with per-pair sigmoid:

```
loss = mean over pairs of log(1 + exp(-y_ij * sim_ij))
y_ij = +1 if matching, -1 otherwise
```

Per-pair loss removes the batch-level normalisation that CLIP requires. SigLIP trains better at small batch sizes and matches or exceeds CLIP at equal data.


# Build your Own

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTower(nn.Module):
    def __init__(self, img_in=128, txt_in=64, emb=64):
        super().__init__()
        self.image_proj = nn.Sequential(
            nn.Linear(img_in, 128),
            nn.ReLU(),
            nn.Linear(128, emb)
        )

        self.text_proj = nn.Sequential(
            nn.Linear(txt_in, 128),
            nn.ReLU(),
            nn.Linear(128, emb)
        )

        # 可学习温度（对数域）：exp(.) ≈ 1/τ；初值 2.6592 ≈ log(1/0.07)
        # 用于放大余弦相似度再做 CE，使正负对更好分：logits = (i @ t.T) * scale
        self.logit_scale = nn.Parameter(torch.ones([]) * 2.6592)

    def forward(self, img_feats, txt_feats):
        i = F.normalize(self.image_proj(img_feats), dim=-1)
        t = F.normalize(self.text_proj(txt_feats), dim=-1)

        return i, t, self.logit_scale.exp()  # 返回正数温度乘数，供外部算相似度矩阵


# Constrastive Loss

In [6]:
def clip_loss(image_emb, text_emb, logit_scale):
    N = image_emb.size(0)
    sim = logit_scale * image_emb @ text_emb.T
    targets = torch.arange(N, device=sim.device)
    l_i = F.cross_entropy(sim, targets)
    l_t = F.cross_entropy(sim.T, targets)
    return (l_i + l_t) / 2

## Zero-shot classifer

In [8]:
@torch.no_grad()
def zero_shot_classify(model, image_feats, class_feats, class_names):
    """
    image_feats: (N, img_in)
    class_feats: (C, txt_in) one averaged embedding per class
    """
    i = F.normalize(model.image_proj(image_feats), dim=-1)
    t = F.normalize(model.text_proj(class_feats), dim=-1)
    sim =  i @ t.T

    pred = sim.argmax(dim=-1)
    return [class_names[p] for p in pred.tolist()]

## Sanity Check

In [ ]:
torch.manual_seed(0)

model = TwoTower()

img = torch.randn(8, 128)
txt = torch.randn(8, 64)

i, t, scale = model(img, txt)
loss = clip_loss(i, t, scale)

print(f"Batch size: {i.size(0)}, loss: {loss.item():.3f}")